In [22]:
import requests
import pandas as pd
import uuid

# Basis configuratie gebaseerd op de technische documentatie
BASE_URL = "https://api.ah.nl"
STORE_ID = "1558"  # Dit is het unieke ID voor AH Woenselse Markt Eindhoven

# Verplichte headers om de AH app na te bootsen
HEADERS = {
    "User-Agent": "Appie/9.28 (iPhone17,3; iPhone; CPU OS 26_1 like Mac OS X)",
    "x-application": "AHWEBSHOP",
    "x-clientname": "appie-ios",
    "x-": "9.28",
    "x-fraud-detection-installation-id": str(uuid.uuid4()), # Een unieke ID per sessie
    "Content-Type": "application/json",
    "Accept": "application/json"
}


In [34]:
def get_ah_token():
    auth_url = f"{BASE_URL}/mobile-auth/v1/auth/token/anonymous"
    payload = {"clientId": "appie-ios"}
    
    response = requests.post(auth_url, json=payload, headers=HEADERS)
    response.raise_for_status() # Geeft een foutmelding als het misgaat
    
    token_data = response.json()
    return token_data['access_token']

# Activeer de sleutel voor alle volgende verzoeken
access_token = get_ah_token()
HEADERS["Authorization"] = f"Bearer {access_token}"
print("Handshake succesvol: Token opgehaald.")

Handshake succesvol: Token opgehaald.


In [24]:
bargain_query = """
query GetBargains($storeId: String!) {
  bargainItems(storeId: $storeId) {
    categoryTitle  # <--- Deze voegt de categorie (zoals Vlees) toe
    product {
      title
      brand
      salesUnitSize
    }
    bargainPrice {
      priceWas
      priceNow
    }
    markdown {
      markdownPercentage
      markdownExpirationDate
    }
    stock
  }
}
"""

def fetch_laatste_kans(store_id):
    url = f"{BASE_URL}/graphql"
    
    graphql_headers = headers.copy()
    graphql_headers.update({
        "x-apollo-operation-name": "GetBargains",
        "x-apollo-operation-type": "query",
        "apollographql-client-name": "nl.ah.Appie-apollo-ios",
        "apollographql-client-version": "9.28-260102201630"
    })
    
    payload = {
        'query': bargain_query, 
        'variables': {'storeId': store_id},
        'operationName': 'GetBargains'
    }
    
    response = requests.post(url, json=payload, headers=graphql_headers)
    data = response.json()
    
    if 'errors' in data:
        print(" GraphQL Foutmelding gevonden:")
        for error in data['errors']:
            print(f" - {error.get('message')}")
        return None
        
    return data.get('data', {}).get('bargainItems')

# Haal de ruwe data opnieuw op
raw_items = fetch_laatste_kans(STORE_ID)

if raw_items:
    print(f"Succes! {len(raw_items)} producten gevonden op de Woenselse Markt.")
else:
    print("Geen data ontvangen. Controleer de output hierboven.")

Succes! 76 producten gevonden op de Woenselse Markt.


In [25]:
# Gebruik json_normalize om geneste velden (zoals product.title) plat te slaan
df_koopjes = pd.json_normalize(raw_items)

# Optioneel: Kolomnamen opschonen voor gemak
df_koopjes.columns = [c.replace('product.', '').replace('bargainPrice.', '').replace('markdown.', '') for c in df_koopjes.columns]

# Sorteer op de hoogste korting
# df = df.sort_values(by='markdownPercentage', ascending=False)

# Toon de live status
display(df_koopjes.head(10))

,categoryTitle,stock,priceWas,priceNow,markdownPercentage,markdownExpirationDate,title,brand,salesUnitSize
0,"Groente, aardappelen",1,0.99,0.30,70,2026-02-04,AH Snoepgroente worteltjes,AH,150 g
1,"Groente, aardappelen",5,1.29,0.97,25,2026-02-05,AH Dille,AH,15 g
2,"Groente, aardappelen",3,1.19,0.89,25,2026-02-05,AH Fijngesneden spitskool,AH,200 g
3,"Groente, aardappelen",2,5.89,4.42,25,2026-02-05,AH Curry madras verspakket,AH,4 pers | 30 min
4,"Groente, aardappelen",2,5.59,4.19,25,2026-02-05,AH Kip piri piri verspakket,AH,4 pers | 35 min
5,"Groente, aardappelen",2,2.19,1.64,25,2026-02-05,AH Gesneden spitskool,AH,400 g
6,"Groente, aardappelen",1,1.59,1.19,25,2026-02-05,AH Macaroni spaghetti groente klein,AH,250 g
7,"Groente, aardappelen",1,1.89,1.42,25,2026-02-05,AH Macaroni spaghetti groente,AH,450 g
8,"Groente, aardappelen",1,1.99,1.49,25,2026-02-05,AH Gele wortel nasi,AH,400 g
9,"Groente, aardappelen",1,1.69,1.27,25,2026-02-05,AH IJsbergsla fijngesneden,AH,200 g


In [36]:
def get_df_bonus():
    token = get_ah_token()
    auth_headers = {**HEADERS, "Authorization": f"Bearer {token}"}
    
    # GraphQL query: 'mainCategory' vervangen door 'category'
    query = """
    query GetNationalBonus {
      bonusPromotions {
        title
        category  # De categorie van de hele promotiegroep
        products {
          title
          brand
          category # De categorie van het specifieke product
          priceV2 {
            now { amount }
            was { amount }
            discount { description }
          }
        }
      }
    }
    """
    
    response = requests.post(f"{BASE_URL}/graphql", json={"query": query}, headers=auth_headers)
    response.raise_for_status()
    raw_data = response.json().get('data', {}).get('bonusPromotions', [])
    
    if not raw_data:
        return pd.DataFrame()

    # We flatten de producten en nemen de categorieën mee
    df = pd.json_normalize(
        raw_data, 
        record_path=['products'], 
        meta=['title', 'category'], # 'category' van de promo toevoegen als meta
        record_prefix='product_',
        meta_prefix='promo_'
    )
    
    # Mapping: Gebruik 'product_category' voor de meest specifieke indeling
    mapping = {
        'product_title': 'Product',
        'product_brand': 'Merk',
        'product_category': 'Categorie',  # Dit veld komt nu wel door
        'product_priceV2.now.amount': 'Prijs_Nu',
        'product_priceV2.was.amount': 'Prijs_Was',
        'product_priceV2.discount.description': 'Bonus_Tekst',
        'promo_title': 'Bonus_Groep'
    }
    
    df_bonus = df.rename(columns=mapping)[list(mapping.values())]
    
    # Berekeningen
    df_bonus['Prijs_Nu'] = pd.to_numeric(df_bonus['Prijs_Nu'], errors='coerce')
    df_bonus['Prijs_Was'] = pd.to_numeric(df_bonus['Prijs_Was'], errors='coerce')
    df_bonus['Korting_Pct'] = ((df_bonus['Prijs_Was'] - df_bonus['Prijs_Nu']) / df_bonus['Prijs_Was'] * 100).round(0)
    
    return df_bonus


df_bonus= get_df_bonus()

In [29]:
# 1. Normaliseer de titels zoals we al deden
df_bonus['Product_clean'] = df_bonus['Product'].str.lower().str.strip()
df_koopjes['title_clean'] = df_koopjes['title'].str.lower().str.strip()

# 2. De Merge
df_double_deals = pd.merge(
    df_bonus, 
    df_koopjes, 
    left_on='Product_clean', 
    right_on='title_clean', 
    how='inner'
)

# 3. FIX: Zet prijzen om naar getallen (errors='coerce' maakt van tekst NaN)
df_double_deals['priceNow'] = pd.to_numeric(df_double_deals['priceNow'], errors='coerce')
df_double_deals['Prijs_Was'] = pd.to_numeric(df_double_deals['Prijs_Was'], errors='coerce')

# 4. Bereken nu de korting (met beveiliging tegen delen door nul)
df_double_deals['Totale_Korting_Percentage'] = (1 - (df_double_deals['priceNow'] / df_double_deals['Prijs_Was'])) * 100

# 5. Filter en toon resultaat
sniper_resultaat = df_double_deals[[
    'Product', 
    'Bonus_Tekst', 
    'markdownPercentage', 
    'Prijs_Was', 
    'priceNow', 
    'Totale_Korting_Percentage'
]].dropna(subset=['Totale_Korting_Percentage']) # Haal ongeldige berekeningen eruit


df_double_deals['Totale_Korting_Percentage'] = df_double_deals['Totale_Korting_Percentage'].round(2)
display(sniper_resultaat.sort_values(by='Totale_Korting_Percentage', ascending=False))

,Product,Bonus_Tekst,markdownPercentage,Prijs_Was,priceNow,Totale_Korting_Percentage
8,AH Stoomsoep tom kha kai,VOOR 4.99,70,6.99,1.50,78.540773
1,Arla Biologisch halfvolle melk,NaN,25,2.39,0.89,62.761506
0,Arla Biologisch halfvolle melk,NaN,25,1.79,0.89,50.279330
7,AH Stoommaaltijd zalm roomsaus,VOOR 4.99,25,7.49,3.74,50.066756
5,AH Lasagne verspakket,1 euro korting,25,5.49,3.37,38.615665
9,AH Stoommaaltijd pasta pesto,VOOR 4.99,25,5.99,3.74,37.562604
2,Arla Biologisch halfvolle melk,NaN,25,1.19,0.89,25.210084


In [30]:
import os
import google.generativeai as genai
from dotenv import load_dotenv

load_dotenv(override=True)

# Haal de key op en maak hem direct schoon
raw_key = os.getenv("GEMINI_API_KEY")
if raw_key:
    # Verwijder spaties, quotes en witregels
    api_key = raw_key.strip().strip('"').strip("'")
    
    genai.configure(api_key=api_key)
    model = genai.GenerativeModel("gemma-3-27b-it")    
    try:
        # Test de verbinding
        response = model.generate_content("Hoi Chef!")
        print("✅ Verbinding geslaagd! De motor draait.")
    except Exception as e:
        print(f"❌ Google zegt nog steeds nee: {e}")
else:
    print("❌ Sleutel niet gevonden in .env")

✅ Verbinding geslaagd! De motor draait.
